# Chess LLM-RL Training (Lightning AI A100)

This notebook is optimized for automated, non-interactive execution. Paste your keys in Cell 1 and run all.

In [ ]:
# 1. CREDENTIALS & CONFIGURATION
import os

# Paste your keys here to avoid interactive prompts
WANDB_KEY = "your_wandb_api_key_here"
HF_TOKEN  = "your_huggingface_token_here"
HF_REPO   = "your-username/chess-llm-rl"

os.environ['WANDB_API_KEY'] = WANDB_KEY
os.environ['HF_TOKEN']      = HF_TOKEN
os.environ['HF_REPO']       = HF_REPO
os.environ['TARGET_GAMES']  = '180'  # Total games to play

In [ ]:
# 2. Install Dependencies & Stockfish (Silent)
print("Installing dependencies...")
!pip install -q chess python-chess pyyaml wandb trl peft unsloth huggingface_hub datasets

print("Setting up Stockfish...")
!bash setup_stockfish.sh

import os, sys, torch
from pathlib import Path

# Ensure the src directory is in the path
CWD = os.getcwd()
if os.path.join(CWD, 'src') not in sys.path:
    sys.path.append(os.path.abspath('src'))

import wandb
from huggingface_hub import login

print("Authenticating...")
if WANDB_KEY != "your_wandb_api_key_here":
    wandb.login(key=WANDB_KEY)
if HF_TOKEN != "your_huggingface_token_here":
    login(token=HF_TOKEN)
print("Authentication successful.")

In [ ]:
# 3. Apply HF Hub Patch (No Source Changes)
import chess_rl.train
from huggingface_hub import HfApi

def patched_save_checkpoint(model, tok, cfg, state, games_seen):
    """Monkey-patch for train.py to add HF Hub support."""
    # 1. Local save
    path = Path(cfg["paths"]["checkpoints"]) / f"ckpt_games_{games_seen:06d}"
    path.mkdir(parents=True, exist_ok=True)
    model.save_pretrained(str(path))
    tok.save_pretrained(str(path))
    import json
    with open(path / "state.json", "w", encoding="utf-8") as f:
        json.dump(state, f, indent=2)
    
    # 2. Upload to HF Hub
    repo = os.environ.get("HF_REPO")
    token = os.environ.get("HF_TOKEN")
    if repo and token and token != "your_huggingface_token_here":
        try:
            api = HfApi(token=token)
            api.create_repo(repo, exist_ok=True, private=True)
            api.upload_folder(
                folder_path=str(path),
                repo_id=repo,
                path_in_repo=path.name,
                commit_message=f"ckpt games={state.get('games_seen')} elo={state.get('elo')}",
            )
            print(f"[hub] pushed checkpoint to {repo}")
        except Exception as e:
            print(f"[hub] push failed: {e}")
    return path

chess_rl.train.save_checkpoint = patched_save_checkpoint
print("HF Hub integration patched into training loop.")

In [ ]:
# 4. Launch Training (auto-resume from latest HF checkpoint if any)
from chess_rl.train import main as train_main
from huggingface_hub import HfApi, snapshot_download

CONFIG = 'config.yaml'

def find_latest_hf_checkpoint(repo: str, token: str):
    """Return (folder_name, games_seen) of the latest ckpt_games_* in the repo, or None."""
    if not repo or not token or token == "your_huggingface_token_here":
        return None
    try:
        api = HfApi(token=token)
        files = api.list_repo_files(repo_id=repo)
    except Exception as e:
        print(f"[hub] could not list repo: {e}")
        return None
    buckets = {}
    for f in files:
        # paths look like "ckpt_games_000056/adapter_model.safetensors"
        parts = f.split("/")
        if len(parts) >= 2 and parts[0].startswith("ckpt_games_"):
            try:
                n = int(parts[0].split("_")[-1])
                buckets.setdefault(parts[0], set()).add(parts[1])
            except ValueError:
                continue
    # require both adapter weights and state.json present
    valid = [name for name, fs in buckets.items()
             if "adapter_model.safetensors" in fs and "state.json" in fs]
    if not valid:
        return None
    latest = max(valid, key=lambda n: int(n.split("_")[-1]))
    return latest, int(latest.split("_")[-1])

RESUME_FROM = None
hit = find_latest_hf_checkpoint(os.environ.get("HF_REPO"), os.environ.get("HF_TOKEN"))
if hit is not None:
    folder, games_seen = hit
    print(f"[hub] resuming from {folder} (games_seen={games_seen})")
    local_root = snapshot_download(
        repo_id=os.environ["HF_REPO"],
        token=os.environ["HF_TOKEN"],
        allow_patterns=f"{folder}/*",
    )
    RESUME_FROM = str(Path(local_root) / folder)
else:
    print("[hub] no prior checkpoint found — starting fresh")

try:
    os.makedirs('checkpoints', exist_ok=True)
    train_main(CONFIG, resume_from=RESUME_FROM)
except Exception as e:
    print(f"Training failed: {e}")
    import traceback
    traceback.print_exc()